# multi_1h_jma_atr_008

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (14).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 14 |
| Detected functions | resample_1hr_custom, compute_jma_indicator, compute_stats, evaluate_params, cy_process_ticker, backtest_trades, calculate_metrics_fast, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Crypto_Data/Binance_Crypto_T7_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2024-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['SOLUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
import pandas as pd
import numpy as np

def compute_jma_indicator(
    df,
    smooth_length=21,
    normalization_period=24,
    upper_threshold=2,
    lower_threshold=-2.25,
    rs_period=10,
    rs_percentile=90,
    tp_mult=1.75,
    sl_mult=1.0
):
    df = df.copy()

    # === JMA Calculation (EMA-style) ===
    price_series = df['close'].values
    alpha = 2.0 / (smooth_length + 1)
    jma = np.empty_like(price_series)
    jma[0] = price_series[0]
    for i in range(1, len(price_series)):
        jma[i] = jma[i - 1] + (price_series[i] - jma[i - 1]) * alpha
    jma_series = pd.Series(jma, index=df.index)

    # === Normalization ===
    rolling_mean = jma_series.rolling(window=normalization_period).mean()
    rolling_std = jma_series.rolling(window=normalization_period).std()
    normalized_jma = (jma_series - rolling_mean) / rolling_std

    # === RS Volatility ===
    safe_open = df['open'].clip(lower=1e-8)
    safe_high = df['high'].clip(lower=1e-8)
    safe_low = df['low'].clip(lower=1e-8)
    safe_close = df['close'].clip(lower=1e-8)

    log_hc = np.log(safe_high / safe_close)
    log_ho = np.log(safe_high / safe_open)
    log_lc = np.log(safe_low / safe_close)
    log_lo = np.log(safe_low / safe_open)

    rs_volatility = (log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()

    # === RS Volatility Threshold ===
    rs_threshold = rs_volatility.rolling(window=normalization_period).quantile(rs_percentile / 100.0)

    # === Signal Generation (Vectorized) ===
    long_signals = (normalized_jma > upper_threshold) & (rs_volatility > rs_threshold)
    short_signals = (normalized_jma < lower_threshold) & (rs_volatility > rs_threshold)

    signals = np.zeros(len(df), dtype=np.int8)
    signals[long_signals] = 1
    signals[short_signals] = 2

    # === TP/SL Calculation (Vectorized) ===
    close_price = df['close'].values
    vol_array = rs_volatility.values

    tp_level = np.full(len(df), np.nan)
    sl_level = np.full(len(df), np.nan)

    long_idx = signals == 1
    short_idx = signals == 2

    tp_level[long_idx] = close_price[long_idx] + vol_array[long_idx] * tp_mult
    sl_level[long_idx] = close_price[long_idx] - vol_array[long_idx] * sl_mult

    tp_level[short_idx] = close_price[short_idx] - vol_array[short_idx] * tp_mult
    sl_level[short_idx] = close_price[short_idx] + vol_array[short_idx] * sl_mult

    # === Attach to DataFrame ===
    df['Signal'] = signals
    df['TP_Level'] = tp_level
    df['SL_Level'] = sl_level
    df['rs_volatility'] = rs_volatility

    return df[['Date', 'Ticker', 'open', 'high', 'low', 'close', 'Signal', 'rs_volatility', 'TP_Level', 'SL_Level']]

# === Ensure datetime ===
if not np.issubdtype(data['Date'].dtype, np.datetime64):
    data['Date'] = pd.to_datetime(data['Date'])

# === Apply per ticker using groupby + apply ===
data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator).reset_index(drop=True)

# === Shift signals to prevent lookahead bias ===
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data.dropna(subset=['Signal'], inplace=True)
data['Signal'] = data['Signal'].astype(np.int8)

# === Final cleaned data ===
data = data.reset_index(drop=True)
data

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Ensure datetime column is properly parsed
data['Date'] = pd.to_datetime(data['Date'])

# Separate Buy and Sell signals
buy_signals = data[data['Signal'] == 1].copy()
sell_signals = data[data['Signal'] == 2].copy()

# Buy signal calculations
buy_signals['return'] = buy_signals['close'] - buy_signals['open']
buy_signals['upside'] = buy_signals['high'] - buy_signals['open']
buy_signals['downside'] = buy_signals['open'] - buy_signals['low']

print("Buy Signal Stats:")
print("Avg Return:", buy_signals['return'].mean())
print("Win Rate:", (buy_signals['return'] > 0).mean())
print("Avg Upside:", buy_signals['upside'].mean())
print("Avg Downside:", buy_signals['downside'].mean())
print("Return Quantiles:\n", buy_signals['return'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

# Sell signal calculations
sell_signals['return'] = sell_signals['open'] - sell_signals['close']
sell_signals['upside'] = sell_signals['open'] - sell_signals['low']
sell_signals['downside'] = sell_signals['high'] - sell_signals['open']

print("\nSell Signal Stats:")
print("Avg Return:", sell_signals['return'].mean())
print("Win Rate:", (sell_signals['return'] > 0).mean())
print("Avg Upside:", sell_signals['upside'].mean())
print("Avg Downside:", sell_signals['downside'].mean())
print("Return Quantiles:\n", sell_signals['return'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

# ================= Plotly Visualizations =================

# 1. Buy Return Histogram
fig1 = px.histogram(buy_signals, x="return", nbins=100, title="Buy Signal Return Distribution")
fig1.show()

# 2. Cumulative Return Curve
buy_signals['cum_return'] = buy_signals['return'].cumsum()
fig2 = px.line(buy_signals, x="Date", y="cum_return", title="Cumulative Return - Buy Signals")
fig2.show()

# 3. Upside vs Downside Box Plot
df_buy = buy_signals.melt(id_vars=['Date'], value_vars=['upside', 'downside'])
fig3 = px.box(df_buy, x='variable', y='value', title='Upside vs Downside - Buy Signals')
fig3.show()

# 4. Daily Win Rate (Buy Signals)
buy_signals['date_only'] = buy_signals['Date'].dt.date
win_rate_daily = buy_signals.groupby('date_only')['return'].apply(lambda x: (x > 0).mean()).reset_index(name='win_rate')
fig4 = px.bar(win_rate_daily, x='date_only', y='win_rate', title='Daily Buy Signal Win Rate')
fig4.show()

# Fix for interval-to-string issue + observed=False warning
buy_signals['hour'] = buy_signals['Date'].dt.hour
buy_signals['quantile'] = pd.qcut(buy_signals['return'], q=10, duplicates='drop').astype(str)

pivot = buy_signals.pivot_table(
    index='quantile',
    columns='hour',
    values='return',
    aggfunc='mean',
    observed=False  # Suppress future warning
)

fig5 = px.imshow(pivot, title='Avg Return Heatmap by Hour & Return Quantile')
fig5.show()

# 6. Drawdown Curve
cum = buy_signals['return'].cumsum()
rolling_max = cum.cummax()
drawdown = cum - rolling_max
fig6 = px.area(x=buy_signals['Date'], y=drawdown, title='Buy Signal Drawdown')
fig6.show()


In [ ]:
from itertools import product
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
import numpy as np
import os

# ----------------------------- #
#         PARAMETER GRID       #
# ----------------------------- #
param_grid = {
    'smooth_length': [5, 10, 14, 21],
    'normalization_period': [24, 48, 56, 72, 96, 120],
    'upper_threshold': [1.0, 1.25, 1.5, 1.75, 2, 2.25, 2.5],
    'lower_threshold': [-1.0, -1.25, -1.5, -1.75, -2.0, -2.25, -2.5],
    'rs_period': [3, 6, 10, 14, 21],
    'rs_percentile': [30, 50, 70, 80, 90],
    'tp_mult': [1],
    'sl_mult': [1]
}
param_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())

# ----------------------------- #
#         CSV LOG FILE         #
# ----------------------------- #
log_csv = "jma_hypertune_log.csv"

# Extended stat fields
stat_fields = ['Avg Return', 'Win Rate', 'Avg Upside', 'Avg Downside', 'Q10', 'Q25', 'Q50', 'Q75', 'Q90']
buy_sell_columns = [f'{side} {field}' for side in ['Buy', 'Sell'] for field in stat_fields]

# Create new CSV with headers if it doesn't exist
if not os.path.exists(log_csv):
    pd.DataFrame(columns=param_names + buy_sell_columns + ['Buy Trades', 'Sell Trades', 'Error']).to_csv(log_csv, index=False)

# ----------------------------- #
#     CUSTOM STAT CALCULATOR   #
# ----------------------------- #
def compute_stats(trades):
    stats = {}
    returns = trades['return']
    stats['Avg Return'] = returns.mean()
    stats['Win Rate'] = (returns > 0).mean()
    stats['Avg Upside'] = returns[returns > 0].mean()
    stats['Avg Downside'] = returns[returns < 0].mean()

    quantiles = returns.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    stats.update({
        'Q10': quantiles.loc[0.10],
        'Q25': quantiles.loc[0.25],
        'Q50': quantiles.loc[0.50],
        'Q75': quantiles.loc[0.75],
        'Q90': quantiles.loc[0.90],
    })
    return stats

# ----------------------------- #
#     PARAMETER EVALUATION     #
# ----------------------------- #
def evaluate_params(combo):
    params = dict(zip(param_names, combo))
    try:
        df = data.copy()

        df = df.groupby('Ticker', group_keys=False).apply(lambda d: compute_jma_indicator(
            d,
            smooth_length=params['smooth_length'],
            normalization_period=params['normalization_period'],
            upper_threshold=params['upper_threshold'],
            lower_threshold=params['lower_threshold'],
            rs_period=params['rs_period'],
            rs_percentile=params['rs_percentile'],
            tp_mult=params['tp_mult'],
            sl_mult=params['sl_mult']
        )).reset_index(drop=True)

        df['Signal'] = df.groupby('Ticker')['Signal'].shift(1)
        df.dropna(subset=['Signal'], inplace=True)
        df['Signal'] = df['Signal'].astype(np.int8)

        buy = df[df['Signal'] == 1].copy()
        sell = df[df['Signal'] == 2].copy()

        buy['return'] = buy['close'] - buy['open']
        sell['return'] = sell['open'] - sell['close']

        buy_stats = compute_stats(buy)
        sell_stats = compute_stats(sell)

        result = {
            **params,
            **{f'Buy {k}': v for k, v in buy_stats.items()},
            **{f'Sell {k}': v for k, v in sell_stats.items()},
            'Buy Trades': len(buy),
            'Sell Trades': len(sell),
            'Error': None
        }

    except Exception as e:
        result = {
            **params,
            **{f'Buy {k}': np.nan for k in stat_fields},
            **{f'Sell {k}': np.nan for k in stat_fields},
            'Buy Trades': 0,
            'Sell Trades': 0,
            'Error': str(e)
        }

    # Append result to CSV
    pd.DataFrame([result]).to_csv(log_csv, mode='a', header=False, index=False)
    return result

# ----------------------------- #
#         HYPERTUNING LOOP     #
# ----------------------------- #
results = []
for combo in tqdm(param_combos, desc="Hypertuning"):
    results.append(evaluate_params(combo))

# ----------------------------- #
#       SHOW TOP RESULTS       #
# ----------------------------- #
results_df = pd.read_csv(log_csv)
top = results_df.sort_values(by='Buy Avg Return', ascending=False).head(10)

print("\n🔝 Top 10 Parameter Sets by Buy Avg Return:")
print(top[['Buy Avg Return', 'Buy Win Rate', 'Buy Q90'] + param_names])


In [ ]:
%load_ext cython

In [ ]:
%%cython
import numpy as np
cimport numpy as np

def cy_process_ticker(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double tp_pct,
    double tsl_pct,
    double entry_esl_pct,
    double universal_sl_pct
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        double entry_price = 0.0
        object entry_time = None
        int direction = -1  # Always short
        double quantity = 0.0
        double cumulative_pnl = 0.0
        object last_exit_time = None
        bint sl_tp_check_active = False
        list trades = []

        double tp_price, trailing_sl, max_fav_price, exit_price
        double pnl, fee, net_pnl, unrealized_loss_pct
        bint tp_hit, sl_hit, esl_hit
        object time
        double open_, high, low, close
        int signal

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        # === ENTRY ===
        if not position_open and time != last_exit_time:
            if signal != 2:  # Only act on short signal
                continue

            direction = -1
            entry_price = open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            position_open = True
            sl_tp_check_active = False

            tp_price = entry_price * (1 - tp_pct)
            trailing_sl = entry_price * (1 + tsl_pct)
            max_fav_price = entry_price

            esl_price = entry_price * (1 + entry_esl_pct)
            esl_hit = (high >= esl_price)

            if esl_hit:
                exit_price = esl_price * (1 + slippage_rate)
                pnl = (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Entry ESL", cumulative_pnl, "short"))
                position_open = False
                last_exit_time = time
                continue

            tp_hit = (low <= tp_price)
            if tp_hit:
                exit_price = tp_price * (1 + slippage_rate)
                pnl = (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Entry TP", cumulative_pnl, "short"))
                position_open = False
                last_exit_time = time
                continue

            sl_tp_check_active = True
            continue

        # === POSITION MANAGEMENT ===
        if position_open:
            if low < max_fav_price:
                max_fav_price = low
                trailing_sl = max_fav_price * (1 + tsl_pct)

            tp_hit = (low <= tp_price)
            sl_hit = (high >= trailing_sl)

            if sl_tp_check_active and sl_hit:
                exit_price = open_ * (1 + slippage_rate)
                pnl = (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Trailing SL", cumulative_pnl, "short"))
                position_open = False
                last_exit_time = time
                continue

            if tp_hit:
                exit_price = tp_price * (1 + slippage_rate)
                pnl = (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Take Profit", cumulative_pnl, "short"))
                position_open = False
                last_exit_time = time
                continue

            unrealized_loss_pct = (entry_price - close) / entry_price
            if unrealized_loss_pct <= -universal_sl_pct:
                exit_price = close * (1 + slippage_rate)
                pnl = (entry_price - exit_price) * quantity
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl
                trades.append((ticker, entry_time, time, entry_price, exit_price, net_pnl, "Universal SL", cumulative_pnl, "short"))
                position_open = False
                last_exit_time = time
                continue

            sl_tp_check_active = True

    return trades


In [ ]:
def backtest_trades(data: pd.DataFrame, tp_pct: float, tsl_pct: float, entry_esl_pct, universal_sl_pct) -> pd.DataFrame:
    from tqdm import tqdm

    initial_capital = 100000
    fee = 0.0004  # 0.04% round trip
    slippage = 0.0005  # 0.05% round trip

    # Ensure datetime format and avoid future warning
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Build per-ticker data matrix with 8 columns
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal", "TP_Level", "SL_Level"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_process_ticker(
            ticker_map[ticker], ticker, initial_capital, fee, slippage, tp_pct, tsl_pct, entry_esl_pct, universal_sl_pct
        )
        all_trades.extend(trades)

    # Create DataFrame from trade results
    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# === Example usage ===
# Run the backtest
tradebook = backtest_trades(data, tp_pct=50, tsl_pct=0.002, entry_esl_pct=0.009, universal_sl_pct=0.009)

# Save or display results
tradebook.to_csv("/content/drive/MyDrive/JMA_Tradebook_Crypto_T5_2022_25_1.csv", index=False)
tradebook


In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np

# === High-Speed Metrics Calculation ===
def calculate_metrics_fast(tradebook):
    if tradebook.empty:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0

    dates = pd.to_datetime(tradebook['Entry Time'].values, errors='coerce')
    pnls = tradebook['PnL'].values

    # Cumulative PnL (Equity Curve)
    equity = np.cumsum(pnls)
    total_profit = equity[-1]

    # Drawdown & Ratio
    roll_max = np.maximum.accumulate(equity)
    drawdown = roll_max - equity
    max_drawdown = np.max(drawdown)
    ratio = total_profit / max_drawdown if max_drawdown > 0 else (float('inf') if total_profit > 0 else 0.0)

    # R² - using formula without sklearn
    x = np.arange(len(equity))
    y = equity
    x_mean = x.mean()
    y_mean = y.mean()
    ss_tot = np.sum((y - y_mean) ** 2)
    ss_res = np.sum((y - ((np.dot(x, y) / np.dot(x, x)) * x)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0

    # Monthly Consistency
    months = dates.to_numpy().astype('datetime64[M]')
    monthly_pnls = {}
    for m, p in zip(months, pnls):
        monthly_pnls[m] = monthly_pnls.get(m, 0.0) + p
    monthly_vals = np.fromiter(monthly_pnls.values(), dtype=float)
    if len(monthly_vals) < 2:
        monthly_consistency = 0.0
    else:
        m_mean = monthly_vals.mean()
        m_std = monthly_vals.std()
        monthly_consistency = m_mean / m_std if m_std > 0 else float('inf')

    # Win Rate and Total Trades
    total_trades = len(pnls)
    win_rate = np.sum(pnls > 0) / total_trades if total_trades > 0 else 0.0

    return ratio, total_profit, r2, monthly_consistency, win_rate, total_trades

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [10, 14, 21],
    'norm_period': [48, 72, 96, 120],
    'upper_threshold': [1, 1.25, 1.5],
    'lower_threshold': [-1, -1.25, -1.5],
    'rs_period': [3, 6, 10],
    'rs_percentile': [30, 50, 70],
    'tp_pct': [30, 50, 70 , 90, 100],
    'sl_pct': [0.002, 0.003, 0.005],
    'entry_esl_pct': [0.005, 0.007, 0.009],
    'universal_sl_pct': [0.005, 0.007, 0.009]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    # Avoid deepcopy unless required
    temp_data = data

    def apply_strategy(group):
        return compute_jma_indicator(
            group,
            smooth_length=param['smooth_length'],
            normalization_period=param['norm_period'],
            upper_threshold=param['upper_threshold'],
            lower_threshold=param['lower_threshold'],
            rs_period=param['rs_period'],
            rs_percentile=param['rs_percentile'],
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        tsl_pct=param['sl_pct'],
        tp_pct=param['tp_pct'],
        entry_esl_pct=param['entry_esl_pct'],
        universal_sl_pct=param['universal_sl_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency, win_rate, total_trades = calculate_metrics_fast(tradebook)

    # Optional: reduce print frequency
    print(f"\n=== [{i}/{len(param_combinations)}] Params ===")
    for k, v in param.items():
        print(f"{k}: {v}")
    print(f"➡️ Profit/Drawdown: {ratio:.4f} | Profit: ₹{total_profit:,.2f} | R²: {r2:.4f} | Monthly Score: {monthly_consistency:.4f} | Win Rate: {win_rate:.2%} | Trades: {total_trades}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency,
        'Win Rate': win_rate,
        'Total Trades': total_trades
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.2 +
    results_df['Equity R2'] * 0.5 +
    results_df['Monthly Consistency'] * 0.3
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())
